In [3]:
from Utils import NER_Utils
import torch
from transformers import RobertaForTokenClassification, RobertaTokenizerFast, TrainingArguments, Trainer
from Reader import obtain_dataset, obtain_label_list

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    device = torch.device('cuda')
print("Current Device:", torch.cuda.current_device(), torch.cuda.get_device_name(torch.cuda.current_device()))

CUDA available: True
Current Device: 0 NVIDIA GeForce RTX 4070 Ti


In [4]:
datasets, label_list, label2id, id2label = obtain_dataset("TempEval3", "BIO")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/22865 [00:00<?, ? examples/s]

Map:   0%|          | 0/3326 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/223 [00:00<?, ? examples/s]

In [5]:
# Load tokenizer and model
model_name = 'roberta-base'
tokenizer = RobertaTokenizerFast.from_pretrained(model_name, add_prefix_space=True, use_fast=True)
model = RobertaForTokenClassification.from_pretrained(model_name, num_labels=len(label_list), label2id=label2id, id2label=id2label)

d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at roberta-base were not used when initializing RobertaForTokenClassification: ['lm_head.layer_norm.weight', 'lm_head.dense.weight', 'lm_head.bias', 'lm_head.layer_norm.bias', 'lm_head.decoder.weight', 'lm_head.dense.bias']
- This IS expected if you are initializing RobertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForS

In [7]:
training_args = TrainingArguments(
    output_dir="./results/EventTimex-NER",
    logging_dir="./logs/EventTimex-NER",
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_steps=100,
    num_train_epochs=1,
    save_total_limit=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

In [8]:
utils = NER_Utils(tokenizer, label_list)
datasets = utils.tokenize_datasets(datasets)

Map:   0%|          | 0/22865 [00:00<?, ? examples/s]

Map:   0%|          | 0/3326 [00:00<?, ? examples/s]

Map:   0%|          | 0/223 [00:00<?, ? examples/s]

In [9]:
datasets

DatasetDict({
    train: Dataset({
        features: ['tokens', 'label', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 22865
    })
    eval: Dataset({
        features: ['tokens', 'label', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3326
    })
    test: Dataset({
        features: ['tokens', 'label', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 223
    })
})

In [10]:
timex3_ner = Trainer(
    model=model,
    args=training_args,
    compute_metrics=utils.compute_metrics,
    data_collator=utils.data_collator,
    tokenizer=tokenizer,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
)

In [11]:
timex3_ner.train()

The following columns in the training set don't have a corresponding argument in `RobertaForTokenClassification.forward` and have been ignored: tokens. If tokens are not expected by `RobertaForTokenClassification.forward`,  you can safely ignore this message.
d:\GeoTKG\venv\Lib\site-packages\transformers\optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 22865
  Num Epochs = 1
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 1430
  Number of trainable parameters = 124063499


  0%|          | 0/1430 [00:00<?, ?it/s]

The following columns in the evaluation set don't have a corresponding argument in `RobertaForTokenClassification.forward` and have been ignored: tokens. If tokens are not expected by `RobertaForTokenClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 3326
  Batch size = 32


{'loss': 0.2242, 'learning_rate': 4.6503496503496505e-05, 'epoch': 0.07}


  0%|          | 0/104 [00:00<?, ?it/s]

...

  0%|          | 0/104 [00:00<?, ?it/s]

{'eval_loss': 0.11044340580701828, 'eval_precision': 0.8704976213848826, 'eval_recall': 0.8901243147247316, 'eval_f1': 0.8802015728792854, 'eval_runtime': 35.9523, 'eval_samples_per_second': 92.512, 'eval_steps_per_second': 2.893, 'epoch': 0.98}




Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from ./results/EventTimex-NER\checkpoint-1000 (score: 0.8778928136419001).
Deleting older checkpoint [results\EventTimex-NER\checkpoint-500] due to args.save_total_limit
Deleting older checkpoint [results\EventTimex-NER\checkpoint-1000] due to args.save_total_limit


{'train_runtime': 1272.7257, 'train_samples_per_second': 17.965, 'train_steps_per_second': 1.124, 'train_loss': 0.0843634146076816, 'epoch': 1.0}


TrainOutput(global_step=1430, training_loss=0.0843634146076816, metrics={'train_runtime': 1272.7257, 'train_samples_per_second': 17.965, 'train_steps_per_second': 1.124, 'train_loss': 0.0843634146076816, 'epoch': 1.0})

In [12]:
timex3_ner.evaluate(datasets["test"])

The following columns in the evaluation set don't have a corresponding argument in `RobertaForTokenClassification.forward` and have been ignored: tokens. If tokens are not expected by `RobertaForTokenClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 223
  Batch size = 32


  0%|          | 0/7 [00:00<?, ?it/s]

{'eval_loss': 0.1480998545885086,
 'eval_precision': 0.8459796149490374,
 'eval_recall': 0.8450226244343891,
 'eval_f1': 0.8455008488964346,
 'eval_runtime': 0.4678,
 'eval_samples_per_second': 476.691,
 'eval_steps_per_second': 14.963,
 'epoch': 1.0}

In [13]:
timex3_ner.save_model("./results/EventTimex-NER")
timex3_ner.tokenizer.save_pretrained("./results/EventTimex-NER")


Saving model checkpoint to ./results/EventTimex-NER
Configuration saved in ./results/EventTimex-NER\config.json
Model weights saved in ./results/EventTimex-NER\pytorch_model.bin
tokenizer config file saved in ./results/EventTimex-NER\tokenizer_config.json
Special tokens file saved in ./results/EventTimex-NER\special_tokens_map.json
tokenizer config file saved in ./results/EventTimex-NER\tokenizer_config.json
Special tokens file saved in ./results/EventTimex-NER\special_tokens_map.json


('./results/EventTimex-NER\\tokenizer_config.json',
 './results/EventTimex-NER\\special_tokens_map.json',
 './results/EventTimex-NER\\vocab.json',
 './results/EventTimex-NER\\merges.txt',
 './results/EventTimex-NER\\added_tokens.json',
 './results/EventTimex-NER\\tokenizer.json')